# 🔤 Treinamento do YOLO 2: Reconhecimento de Caracteres de Placas (UFPR-ALPR)
### Objetivo: Treinar um modelo YOLO especializado para ler diretamente as letras (A-Z) e números (0-9) das placas brasileiras

Este modelo substitui 100% qualquer OCR genérico, eliminando confusões entre:
- `W` vs `H`
- `1` vs `2`
- `P` vs `2` / `Z`
- Letras com vinil desgastado ou descascado

## 1. Verificação da GPU

In [ ]:
!nvidia-smi

## 2. Instalação das Dependências

In [ ]:
!pip install -q ultralytics pillow
print("✅ Ultralytics instalado!")

## 3. Download e Descompactação Rápida do Dataset UFPR (se ainda não existir)

In [ ]:
import os

ZIP_DESTINO = "/content/UFPR-ALPR.zip"
PASTA_EXTRACAO = "/content/ufpr_raw"

if not os.path.exists(PASTA_EXTRACAO) and not os.path.exists("/content/UFPR-ALPR dataset"):
    !apt-get install -y aria2 > /dev/null 2>&1
    print("🚀 Baixando dataset oficial UFPR...")
    !aria2c -x 16 -s 16 -j 16 -k 1M "https://www.inf.ufpr.br/vri/databases/yj4Iu2-UFPR-ALPR.zip" -d /content -o UFPR-ALPR.zip
    print("📦 Descompactando...")
    !unzip -q /content/UFPR-ALPR*.zip -d /content/ufpr_raw
    !rm -f /content/UFPR-ALPR*.zip
    print("✅ Descompactação concluída!")
else:
    print("✅ Dataset já presente no ambiente!")

## 4. Geração do Dataset de Caracteres das Placas (Crops + Anotações de Letras/Números)
Para cada uma das 4.500 placas, recortamos a placa original e anotamos a posição e classe de cada um dos 7 caracteres (`0-9` e `A-Z`).

In [ ]:
import os
import glob
import shutil
import random
import re
import cv2

# 36 Classes: 0-9 e A-Z
CLASSES = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',
    'U', 'V', 'W', 'X', 'Y', 'Z'
]
CHAR_TO_ID = {c: i for i, c in enumerate(CLASSES)}

DATASET_CHARS = "/content/dataset_yolo_caracteres"
if os.path.exists(DATASET_CHARS):
    shutil.rmtree(DATASET_CHARS)

os.makedirs(os.path.join(DATASET_CHARS, "train/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_CHARS, "train/labels"), exist_ok=True)
os.makedirs(os.path.join(DATASET_CHARS, "val/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_CHARS, "val/labels"), exist_ok=True)

todos_txt = glob.glob("/content/**/track*.txt", recursive=True)
print(f"📄 Anotações de veículos encontradas: {len(todos_txt)}")

amostras_placas = []
for txt_path in todos_txt:
    base = txt_path.rsplit('.', 1)[0]
    img_path = base + ".png" if os.path.exists(base + ".png") else base + ".jpg"
    if not os.path.exists(img_path):
        continue
        
    with open(txt_path, 'r', encoding='latin-1', errors='ignore') as f:
        linhas = f.readlines()
        
    # 1. Encontra a caixa da placa
    xmin_p, ymin_p, xmax_p, ymax_p = None, None, None, None
    caracteres_info = []
    
    for l in linhas:
        l_limpa = l.strip()
        if "corners:" in l_limpa.lower() or "position_plate:" in l_limpa.lower():
            nums = [float(n) for n in re.findall(r"[-+]?(?:\d*\.\d+|\d+)", l_limpa.split(":")[-1])]
            if len(nums) == 8:
                xs, ys = nums[0::2], nums[1::2]
                xmin_p, xmax_p = int(min(xs)), int(max(xs))
                ymin_p, ymax_p = int(min(ys)), int(max(ys))
            elif len(nums) >= 4:
                x, y, w, h = nums[:4]
                xmin_p, ymin_p = int(x), int(y)
                xmax_p, ymax_p = int(x + w), int(y + h)
                
        # 2. Encontra as posições dos caracteres
        if re.match(r"char\s*\d*\s*:", l_limpa.lower()):
            partes = l_limpa.split(":")[-1].strip().split()
            if len(partes) >= 5:
                char_val = partes[0].upper()
                try:
                    cx, cy, cw, ch = map(float, partes[1:5])
                    if char_val in CHAR_TO_ID:
                        caracteres_info.append((char_val, cx, cy, cw, ch))
                except Exception:
                    pass
                    
    if xmin_p is not None and caracteres_info:
        amostras_placas.append((img_path, (xmin_p, ymin_p, xmax_p, ymax_p), caracteres_info))

print(f"✅ Total de {len(amostras_placas)} placas com caracteres anotados!")

# Divide em 80% Treino e 20% Validação
random.seed(42)
random.shuffle(amostras_placas)
corte = int(len(amostras_placas) * 0.8)
splits = {
    "train": amostras_placas[:corte],
    "val": amostras_placas[corte:]
}

for split_nome, lista in splits.items():
    salvos = 0
    for img_path, (px1, py1, px2, py2), chars in lista:
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        # Adiciona pequena margem de segurança ao redor da placa
        ih, iw = img.shape[:2]
        pad_w = int((px2 - px1) * 0.05)
        pad_h = int((py2 - py1) * 0.05)
        x1 = max(0, px1 - pad_w)
        y1 = max(0, py1 - pad_h)
        x2 = min(iw, px2 + pad_w)
        y2 = min(ih, py2 + pad_h)
        
        crop_placa = img[y1:y2, x1:x2]
        if crop_placa.size == 0:
            continue
            
        ch_h, ch_w = crop_placa.shape[:2]
        yolo_chars = []
        
        for c_val, cx, cy, cw, ch in chars:
            # Converte coordenadas para o espaço local do crop da placa
            rel_x = cx - x1
            rel_y = cy - y1
            if rel_x >= 0 and rel_y >= 0 and (rel_x + cw) <= ch_w and (rel_y + ch) <= ch_h:
                cid = CHAR_TO_ID[c_val]
                x_center = (rel_x + cw / 2.0) / ch_w
                y_center = (rel_y + ch / 2.0) / ch_h
                norm_w = cw / ch_w
                norm_h = ch / ch_h
                yolo_chars.append(f"{cid} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
                
        if yolo_chars:
            base_id = os.path.basename(img_path).rsplit('.', 1)[0]
            cv2.imwrite(os.path.join(DATASET_CHARS, f"{split_nome}/images", f"{base_id}.png"), crop_placa)
            with open(os.path.join(DATASET_CHARS, f"{split_nome}/labels", f"{base_id}.txt"), "w") as f_lbl:
                f_lbl.write("\n".join(yolo_chars))
            salvos += 1
            
    print(f"✅ Split {split_nome}: {salvos} placas com caracteres processadas!")

# Cria arquivo data.yaml com as 36 classes
yaml_lines = [
    f"path: {DATASET_CHARS}",
    "train: train/images",
    "val: val/images",
    "names:"
]
for i, c in enumerate(CLASSES):
    yaml_lines.append(f"  {i}: '{c}'")

yaml_path = os.path.join(DATASET_CHARS, "data.yaml")
with open(yaml_path, "w") as f:
    f.write("\n".join(yaml_lines))

print(f"🎉 data.yaml criado para 36 classes de caracteres: {yaml_path}")

## 5. Treinamento do YOLO 2 de Caracteres (Rápido e Preciso na GPU T4)

In [ ]:
from ultralytics import YOLO
from google.colab import files

modelo_chars = YOLO("yolov8n.pt")

resultados = modelo_chars.train(
    data=os.path.join(DATASET_CHARS, "data.yaml"),
    epochs=50,
    imgsz=320,
    batch=32,
    workers=4,
    amp=True,
    patience=15,
    save=True,
    name="yolo_caracteres_placas"
)

print("🎉 Treinamento do YOLO de Caracteres finalizado com sucesso!")

# Download automático do modelo de caracteres treinado
caminho_pesos = "runs/detect/yolo_caracteres_placas/weights/best.pt"
if os.path.exists(caminho_pesos):
    print("⬇️ Baixando modelo de caracteres (yolo_caracteres.pt)...")
    # Renomeia para facilitar identificação
    destino_final = "/content/yolo_caracteres.pt"
    shutil.copy2(caminho_pesos, destino_final)
    files.download(destino_final)
else:
    print("❌ Arquivo best.pt não encontrado.")